# Dynamic Cheatsheet — variante Cumulative (fiel a Suzgun et al. 2025)

**Ref.:** Suzgun et al. (2025), *Dynamic Cheatsheet: Test-Time Learning with Adaptive Memory*, arXiv:2504.07952.

Mecanismo **DC-Cumulative**:
- **Generator:** recomienda usando una **cheatsheet** de texto libre (memoria persistente).
- **Curator (mismo LLM, prompt distinto):** tras responder, **reescribe** la cheatsheet con
  estrategias transferibles. **Label-free** (no usa el GT) — DC es auto-supervisado.
- Memoria = un unico bloque que se **regenera/condensa** en cada paso (el 'monolithic rewrite'
  que ACE critica -> contraste experimental).

**Adaptacion:** construimos la cheatsheet en un warmup sobre train (label-free) y la
**congelamos** para test, por comparabilidad con ACE-offline.

**Consistencia con el proyecto:** datos y metricas desde `utils` (mismo `build_context`,
soft-matching: Recall@1/@5, MRR, Novelty, BERTScore; threshold=0.90, gt_field=`gt_accepted`).

## 0. Dependencias

In [1]:
# === Dependencias (correr una vez) ===
!pip install -q transformers accelerate bitsandbytes sentence-transformers rapidfuzz evaluate bert_score
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.3 MB/s eta 0:00:00
GPU: Tesla T4


## 1. Montar Drive y ubicarse en la carpeta del proyecto

In [2]:
# === Drive: montar y posicionarse en la carpeta del proyecto ===============
# La carpeta esta en "Compartidos conmigo", que Colab NO monta directamente.
# Una sola vez en la web de Drive: click derecho sobre "Proyecto RecSys" ->
# Organizar -> Anadir acceso directo -> "Mi unidad" (es solo un puntero).
import os, sys
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/Proyecto RecSys"
assert os.path.isdir(PROJECT_DIR), (
    "No existe " + PROJECT_DIR + ". Crea el acceso directo en Mi unidad.")
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
PATH = PROJECT_DIR + "/"
print("Trabajando en:", os.getcwd())
print("Archivos:", [f for f in os.listdir() if f.startswith("utils")])

Mounted at /content/drive
Trabajando en: /content/drive/.shortcut-targets-by-id/1DVeLSUjM_ZgsdoEI3aeO7kyHJTdoFuLR/Proyecto RecSys
Archivos: ['utils.ipynb', 'utils.py']


## 2. utils + datos (sin fuga)

In [3]:
# === Bootstrap de utils + carga de datos ===================================
import os, json

if not os.path.exists("utils.py"):
    if os.path.exists("utils.ipynb"):
        _nb = json.load(open("utils.ipynb", encoding="utf-8"))
        with open("utils.py", "w", encoding="utf-8") as f:
            for c in _nb["cells"]:
                if c["cell_type"] == "code":
                    f.write("".join(c["source"]) + "\n\n")
        print("utils.py generado desde utils.ipynb")
    else:
        raise FileNotFoundError("No encuentro utils.ipynb ni utils.py en " + os.getcwd())

from utils import (
    download_dataset, load_parsed, sample_conversations,
    build_context, build_train_freq, build_recommender_references,
    run_full_evaluation,
)

DATASET = "pearl"            # cambiar a "pearl" para correr en PEARL
paths = download_dataset(DATASET, splits=("train", "test"))
train_parsed = load_parsed(paths["train"], dataset=DATASET)
test_parsed  = load_parsed(paths["test"], dataset=DATASET)
print(f"Train: {len(train_parsed)} | Test: {len(test_parsed)}")

# --- Sampling reproducible (mismo seed para todos los metodos) ---
# DC es label-free: el warmup NO usa el GT, pero sampleamos igual para que el
# subconjunto de warmup sea el mismo que el de los demas metodos.
N_WARMUP = 100
N_EVAL   = 300
GT_FIELD = "gt_accepted"
warmup_sample = sample_conversations(train_parsed, n=N_WARMUP, seed=42, gt_field=GT_FIELD)
eval_sample   = sample_conversations(test_parsed,  n=N_EVAL,   seed=42, gt_field=GT_FIELD)
print(f"Warmup: {len(warmup_sample)} | Eval: {len(eval_sample)}")

pearl_train.json ya existe, omitiendo descarga.
pearl_test.json ya existe, omitiendo descarga.
Train: 50000 | Test: 2277
Sampled 100 conversaciones con gt_accepted no vacío (seed=42)
Sampled 300 conversaciones con gt_accepted no vacío (seed=42)
Warmup: 100 | Eval: 300


## 3. Modelo base — Qwen3-8B 4-bit

In [4]:
# === Modelo base: Qwen3-8B 4-bit (cabe en T4) ==============================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, json, re

MODEL_ID = "Qwen/Qwen3-8B"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto")

def call_llm(messages, max_tokens=220):
    """Inferencia determinista (greedy, sin bloque <think>)."""
    text = tokenizer.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
    inp = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False)
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

def parse_json(raw):
    """Extrae el primer objeto JSON de la salida del modelo."""
    try:
        return json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group())
    except Exception:
        return {}

def dedup_titles(titles):
    """Quita titulos duplicados preservando orden (normaliza sin anio)."""
    seen, out = set(), []
    for t in titles:
        k = t.split(" (")[0].strip().lower()
        if k and k not in seen:
            seen.add(k); out.append(t)
    return out


print("Modelo cargado.")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo cargado.


## 4. Artefactos de métricas

In [5]:
# === Embeddings + artefactos para Novelty/BERTScore ========================
from sentence_transformers import SentenceTransformer

device_embed = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2", device=device_embed)

train_freq, n_train = build_train_freq(paths["train"], dataset=DATASET)
human_refs = build_recommender_references(paths["test"], dataset=DATASET)

THRESHOLD = 0.90   # identico al de las metricas (utils)
print("Resultados ->", PATH)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Train freq construido: 9303 títulos únicos en 50000 diálogos
Referencias construidas: 2277 diálogos
Resultados -> /content/drive/MyDrive/Proyecto RecSys/


## 5. Generator + Curator

In [6]:
# === Dynamic Cheatsheet (Cumulative) =======================================
import time
CURATE_EVERY = 1     # DC-Cu cura cada paso; subir (3-5) si el computo aprieta
EMPTY_CS = "(The cheatsheet is currently empty.)"

# --- Generator (prompt recomendador acordado por el grupo) ---
# message: explica la 1a recomendacion con un motivo y menciona las otras 4.
def dc_generate(dialogue, cheatsheet, max_tokens=320):
    system = (
        "You are a conversational movie recommender. You may use the CHEATSHEET of "
        "strategies learned from previous cases.\n\n"
        f"CHEATSHEET:\n{cheatsheet}\n\n"
        "CRITICAL RULE: Under NO circumstances may you recommend a movie that "
        "has already been mentioned by ANYONE (User or Assistant) anywhere in "
        "the conversation history. Every one of your 5 recommendations must be "
        "a movie NOT YET discussed in this conversation — genuinely new "
        "suggestions the user hasn't heard yet.\n\n"
        "Recommend exactly 5 movies, best match first. Include the release "
        "year formatted exactly as 'Title (Year)'.\n"
        "Respond ONLY with valid JSON:\n"
        '{"message": "I recommend [Title (Year)] because [brief reason]. '
        'You might also enjoy [T2], [T3], [T4], and [T5].", '
        '"structured_recommendations": ["Title (Year)", "T2", "T3", "T4", "T5"]}\n'
        "No text outside the JSON."
    )
    raw = call_llm([{"role": "system", "content": system},
                    {"role": "user", "content": build_context(dialogue)}], max_tokens=max_tokens)
    data = parse_json(raw)
    recs = dedup_titles(data.get("structured_recommendations", []) or [])
    return recs, data.get("message", ""), raw

# --- Curator (LABEL-FREE: no ve el GT) ---
def dc_curate(context, answer, cheatsheet, max_tokens=400):
    system = (
        "You are the CURATOR of an evolving CHEATSHEET for a conversational movie "
        "recommender. Given a NEW case and the recommender's own answer, output an "
        "UPDATED cheatsheet of concise, transferable strategies: genre/tone mappings, "
        "how to read user cues, and common pitfalls.\n"
        "Guidelines:\n"
        "- Keep it compact: at most ~20 short bullet lines.\n"
        "- Merge duplicates and drop low-value notes.\n"
        "- Do NOT store one-off answers or specific titles as facts; store reusable tactics.\n"
        "- Output ONLY the full updated cheatsheet text (no preamble)."
    )
    user = (f"CURRENT CHEATSHEET:\n{cheatsheet}\n\nNEW CASE — user request:\n{context}\n\n"
            f"Recommender's answer:\n{answer}\n\nUPDATED CHEATSHEET:")
    new_cs = call_llm([{"role": "system", "content": system},
                       {"role": "user", "content": user}], max_tokens=max_tokens).strip()
    return new_cs if len(new_cs) > 10 else cheatsheet

## 6. Warmup — construir la cheatsheet (label-free)

In [8]:
# === Warmup sobre train: cheatsheet acumulativa (label-free) ===============
def dc_warmup(sample, save_path=None):
    cheatsheet = EMPTY_CS
    t0 = time.time()
    for i, d in enumerate(sample):
        movies, _, _ = dc_generate(d, cheatsheet)
        if (i % CURATE_EVERY) == 0:
            cheatsheet = dc_curate(build_context(d), movies, cheatsheet)
        if (i + 1) % 25 == 0:
            print(f"[{i+1}/{len(sample)}] |cheatsheet|={len(cheatsheet)} chars | "
                  f"ETA {((time.time()-t0)/(i+1))*(len(sample)-i-1)/60:.1f} min")
            if save_path: open(save_path, "w").write(cheatsheet)
    return cheatsheet

_frozen = PATH + f"dc_cheatsheet_frozen_{DATASET}.txt"
if os.path.exists(_frozen):
    cheatsheet = open(_frozen).read()
    print(f"Cheatsheet congelada cargada ({len(cheatsheet)} chars) — se omite el warmup.")
else:
    cheatsheet = dc_warmup(warmup_sample, save_path=PATH + f"dc_cheatsheet_{DATASET}.txt")
    open(_frozen, "w").write(cheatsheet)
    print("\n--- CHEATSHEET FINAL ---\n", cheatsheet[:1500])

Cheatsheet congelada cargada (1842 chars) — se omite el warmup.


## 7. Evaluación (cheatsheet congelada)

In [9]:
# === Evaluacion en test con cheatsheet congelada ===========================
# idx = original_idx para alinear con test_parsed (GT) y human_refs (BERTScore).
def dc_eval(sample, cheatsheet, out_path, save_every=25):
    outs, done = [], set()
    if os.path.exists(out_path):
        outs = json.load(open(out_path)); done = {o["sample_pos"] for o in outs}
    t0 = time.time()
    for pos, d in enumerate(sample):
        if pos in done:
            continue
        movies, msg, raw = dc_generate(d, cheatsheet)
        outs.append({"sample_pos": pos, "idx": d["original_idx"],
                     "predicted": movies, "message": msg, "raw": raw})
        n = len(outs)
        if n % save_every == 0:
            json.dump(outs, open(out_path, "w"))
            eta = ((time.time()-t0)/max(n-len(done),1))*(len(sample)-n)/60
            print(f"{n}/{len(sample)} | ETA {eta:.1f} min")
    json.dump(outs, open(out_path, "w"))
    return outs

out_dc = dc_eval(eval_sample, cheatsheet, PATH + f"dc_results_{DATASET}.json")

print("\n=== Dynamic Cheatsheet — metricas (soft-matching) ===")
metrics_dc = run_full_evaluation(
    out_dc, test_parsed, EMBED_MODEL,
    train_freq, n_train, human_refs,
    threshold=THRESHOLD, gt_field=GT_FIELD, k_list=(1, 5),
)
json.dump(metrics_dc, open(PATH + f"dc_metrics_{DATASET}.json", "w"), indent=2)
print(metrics_dc)

275/300 | ETA 7.1 min
300/300 | ETA 0.0 min

=== Dynamic Cheatsheet — metricas (soft-matching) ===
  EVALUACIÓN COMPLETA

── Recall & MRR (soft-matching) ──
Evaluable: 300 / 300
  Recall@1: 0.0133
  Recall@5: 0.0333
  MRR:      0.0214

── Novelty ──
  Novelty:  10.59 bits (norm: 0.679)
  No vistas en train: 19.4% (266/1373)

── BERTScore ──


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  BERTScore (279/300): P=0.8681  R=0.8831  F1=0.8754

{'Recall@1': 0.0133, 'Recall@5': 0.0333, 'MRR': 0.0214, 'n_evaluable': 279, 'novelty_mean': 10.5947, 'novelty_norm': 0.6787, 'unseen_rate': 0.1937, 'n_dialogos': 279, 'n_recs': 1373, 'precision': 0.8681, 'recall': 0.8831, 'f1': 0.8754, 'n_total': 300}
